## tl;dr
8시간 기본 한도 + HP 최대 2시간, MP 최대 +8%p, DEX·CHA 최대 15%가 후보군 내 잠정 추천입니다. 기존보다 모든 직업 격차가 줄어든 것은 아닙니다. REPORT.md를 함께 읽으세요.

## Context & Methods
운영 DB·앱·네트워크를 사용하지 않습니다. 실제 Kotlin 엔진 성장 경로를 로컬에서 재현했습니다.
### Key Assumptions
8시간은 기본 오프라인 한도, 전경 충전 완료는 12분. 접속 분포 가중치는 없습니다. 후보 탐색 2개 시드, 별도 검증 4개 시드/직업. 노트북 코드 셀은 일반 Python으로 순차 실행했지만 Jupyter 커널 검증은 미실행입니다.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools/analysis/stat_bonus_balance.py').exists())
audit = root / 'docs/audits/2026-08-31-stat-bonus-balance'
sys.path.insert(0, str(root / 'tools/analysis'))
print('Source-backed local artifacts:', audit.name)


Source-backed local artifacts: 2026-08-31-stat-bonus-balance


## Data
원시 경로의 단위는 직업×시드×도달 레벨입니다. 아래 10행은 기준 경로의 결정적 미리보기입니다.

In [2]:
base = pd.read_csv(audit / 'holdout_baseline.csv')
assert len(base) == 288 and not base.duplicated(['class','seed_index','level']).any()
print(base[['class','seed_index','level','hp','mp','tales']].head(10).to_string(index=False))


  class  seed_index  level   hp   mp  tales
WARRIOR           2      1    9    4      0
WARRIOR           2      5   33   37      1
WARRIOR           2     10   84   84      3
WARRIOR           2     20  204  173      7
WARRIOR           2     30  408  294     13
WARRIOR           2     40  707  452     21
WARRIOR           2     50 1117  659     33
WARRIOR           2     60 1725  941     49
WARRIOR           2     70 2537 1294     67
WARRIOR           2     80 3590 1722     87


## Results
저장된 후보 1위와 추천 파라미터가 일치하는지, 독립 검증과 계산식이 다시 통과하는지 확인합니다.

In [3]:
grid = pd.read_csv(audit / 'candidate_grid.csv')
parameters = json.loads((audit / 'selected_parameters.json').read_text())
assert len(grid) == 75264
assert all(abs(grid.iloc[0][k]-v)<1e-12 for k,v in parameters.items())
from validate_stat_bonus_balance import main as validate_study
validate_study()


Validation: Share with caveats control= True
Integration bound Lv100 0.14752144157035296
  variant  level  gap_h  session_min  progress_gap_pct  sale_gap_pct
baseline8     20    8.0         12.0             1.858         1.865
baseline8     20   12.0         12.0             1.858         1.865
baseline8     50    8.0         12.0             1.191         1.248
baseline8     50   12.0         12.0             1.191         1.248
baseline8    100    8.0         12.0             0.500         0.455
baseline8    100   12.0         12.0             0.500         0.455
    old12     20    8.0         12.0             1.858         1.865
    old12     20   12.0         12.0             1.858         1.865
    old12     50    8.0         12.0             1.191         1.248
    old12     50   12.0         12.0             1.191         1.248
    old12    100    8.0         12.0             0.500         0.455
    old12    100   12.0         12.0             0.500         0.455
 selected     

In [4]:
effects = pd.read_csv(audit / 'recommended_effects.csv')
print(effects[effects.level == 100][['class','cap_h','raw_proc_pct','search_s','sale_bonus_pct']].round(3).to_string(index=False))


  class  cap_h  raw_proc_pct  search_s  sale_bonus_pct
 CLERIC  9.772        27.227     4.552          10.781
   MAGE  9.772        27.227     4.552          10.781
PALADIN  9.768        26.249     4.562          13.379
 RANGER  9.788        26.918     4.382          10.819
  ROGUE  9.784        26.294     4.384          10.402
WARRIOR  9.905        26.247     4.562          10.725


## Takeaways
공식은 보너스가 존재한다는 제약 하에 선택됐습니다. 실제 사용자 행동, 무한 레벨 밸런스, 운영 정책 검증이 아닙니다. 8시간이 절대 상한이거나 12분 충전 정책을 바꾸면 다시 계산해야 합니다. 코드·CSV·source SHA-256을 함께 보관하세요.